# 📘 Notebook 7: Conditional Probability, Independence & the Law of Total Probability

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐ (Intermediate)

---

## 🏠 Why Does This Matter?

Suppose you know a house has **4 bedrooms**. Does that change your estimate of its price?  
Of course it does — you'd expect it to be more expensive!

**Conditional probability** formalizes exactly this: "What is P(expensive | has 4 bedrooms)?"  
The `|` means "given that we know..."

This appears everywhere in ML:
- **Classification:** P(class | input features) — the output of every classifier
- **Language models:** P(next word | previous words)
- **Naive Bayes:** P(spam | words in email)
- **Causal reasoning:** Does X cause Y, or are they just correlated?

---

## 📚 Prerequisites
- Notebook 6 (sample spaces, events, Kolmogorov axioms)

---

## Part 1: Conditional Probability

### Plain English First

**Scenario:** You're searching for an expensive house (>$400k) in your dataset.  
Someone tells you: "This house has 4+ bedrooms." Does this new information change your guess?

**Before the information:** P(expensive) = fraction of ALL houses that are expensive  
**After the information:** P(expensive | 4+ bedrooms) = fraction of 4+-bedroom houses that are expensive

The information **zoomed in** your focus from all houses to just the large ones.  
Within that smaller group, what fraction is expensive?

### The Formula

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

Plain English: "Of all the cases where B happened, what fraction also had A?"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────
# Re-create the house dataset from Notebook 6
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N = 100_000

log_prices = np.random.normal(loc=np.log(300_000), scale=0.5, size=N)
prices     = np.exp(log_prices)
bedrooms   = np.random.choice([1,2,3,4,5,6], size=N,
                               p=[0.05, 0.25, 0.40, 0.20, 0.08, 0.02])

# ─── Define events ─────────────────────────────────────────────────
A = prices > 400_000    # Event A: expensive (>$400k)
B = bedrooms >= 4       # Event B: large house (4+ bedrooms)
C = prices < 200_000    # Event C: cheap (<$200k)

def P(event):            return np.mean(event)
def P_given(event_A, event_B):   # P(A | B) = P(A ∩ B) / P(B)
    return np.mean(event_A[event_B])   # mean of A restricted to where B is True


# ─── Compute conditional probabilities ────────────────────────────
print("CONDITIONAL PROBABILITY: House Price Given Bedroom Count")
print("=" * 60)
print()
print(f"Unconditional probabilities:")
print(f"  P(expensive)   = {P(A):.4f}  ({P(A)*100:.1f}% of all houses)")
print(f"  P(large house) = {P(B):.4f}  ({P(B)*100:.1f}% of all houses)")
print()
print(f"Conditional probabilities:")
print(f"  P(expensive | large house)     = {P_given(A, B):.4f}")
print(f"  P(expensive | small house)     = {P_given(A, ~B):.4f}")
print()
print(f"Interpretation:")
print(f"  Without knowing bedrooms: {P(A)*100:.1f}% chance of expensive")
print(f"  Knowing house is LARGE:   {P_given(A,B)*100:.1f}% chance of expensive")
print(f"  Knowing house is SMALL:   {P_given(A,~B)*100:.1f}% chance of expensive")
print()
print("Knowing the bedroom count STRONGLY changes the probability!")
print("This is why features are useful for prediction.")

# ─── Verify the formula: P(A|B) = P(A∩B) / P(B) ──────────────────
print()
print("Formula verification: P(A|B) = P(A∩B) / P(B)")
print(f"  P(A∩B) = {P(A & B):.4f}   (expensive AND large)")
print(f"  P(B)   = {P(B):.4f}   (large house)")
print(f"  Ratio  = {P(A & B)/P(B):.4f}   ← should match P(expensive|large)")
print(f"  Direct = {P_given(A, B):.4f}   ← matches! ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize: conditioning = zooming into a subgroup
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bins = np.linspace(0, 1_500_000, 60)

# All houses
axes[0].hist(prices, bins=bins, density=True, color='steelblue', alpha=0.8)
axes[0].axvline(400_000, color='red', linewidth=2, linestyle='--', label='$400k threshold')
axes[0].set_title(f'ALL houses\nP(expensive) = {P(A):.3f}', fontsize=11)
axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Conditioned on: large house (≥4 bedrooms)
axes[1].hist(prices[B], bins=bins, density=True, color='green', alpha=0.8)
axes[1].axvline(400_000, color='red', linewidth=2, linestyle='--', label='$400k threshold')
axes[1].set_title(f'LARGE houses (4+ bedrooms)\nP(expensive | large) = {P_given(A,B):.3f}\n'
                  '← shifted RIGHT (more expensive)', fontsize=11)
axes[1].set_xlabel('Price ($)'); axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

# Conditioned on: small house (<4 bedrooms)
axes[2].hist(prices[~B], bins=bins, density=True, color='orange', alpha=0.8)
axes[2].axvline(400_000, color='red', linewidth=2, linestyle='--', label='$400k threshold')
axes[2].set_title(f'SMALL houses (<4 bedrooms)\nP(expensive | small) = {P_given(A,~B):.3f}\n'
                  '← shifted LEFT (less expensive)', fontsize=11)
axes[2].set_xlabel('Price ($)'); axes[2].set_ylabel('Density')
axes[2].legend(fontsize=9)

plt.suptitle('Conditional Probability = Zooming Into a Subgroup\n'
             'Knowing bedroom count dramatically changes the price distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: Statistical Independence

### Plain English First

Events A and B are **independent** if knowing B tells you **nothing** about A.

Example: Does knowing a house is expensive tell you anything about which city it was listed in first?  
Probably not if cities are equally distributed — then price and listing-city are independent.

**Test for independence:**
> P(A|B) = P(A)  ← knowing B doesn't change P(A)

This is equivalent to:
$$P(A \cap B) = P(A) \cdot P(B)$$

Plain English: "The probability of both happening = product of individual probabilities"  
(like rolling two dice — getting a 6 on die 1 doesn't affect die 2)

**Why independence matters in ML:**
- **Naive Bayes** classifier assumes features are conditionally independent given the class
- **i.i.d. assumption:** training examples are assumed independent and identically distributed
- **Dropout:** randomly drops neurons independently to prevent co-adaptation

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Test for independence: price vs bedrooms (NOT independent)
#                        price vs a truly random attribute (independent)
# ─────────────────────────────────────────────────────────────────

# Create a truly independent variable: random coin flip per house
# This has no relationship with price
coin_flip = np.random.random(N) < 0.5   # 50/50 random, independent of everything

print("TESTING INDEPENDENCE")
print("=" * 65)

# Test 1: Price vs Bedrooms (should NOT be independent)
print("\nTest 1: Price vs Bedrooms")
print(f"  P(expensive)          = {P(A):.4f}")
print(f"  P(expensive | large)  = {P_given(A, B):.4f}")
print(f"  Difference            = {abs(P(A) - P_given(A,B)):.4f}  ← BIG difference → NOT independent")
print(f"  P(A)×P(B)             = {P(A)*P(B):.4f}")
print(f"  P(A∩B) directly       = {P(A&B):.4f}")
print(f"  Equal?                = {abs(P(A)*P(B) - P(A&B)) < 0.005}  (NO → dependent)")

# Test 2: Price vs random coin flip (should be independent)
print("\nTest 2: Price vs Random Coin Flip")
print(f"  P(expensive)                  = {P(A):.4f}")
print(f"  P(expensive | coin=heads)     = {P_given(A, coin_flip):.4f}")
print(f"  Difference                    = {abs(P(A) - P_given(A, coin_flip)):.4f}  ← tiny → independent")
print(f"  P(A)×P(coin)                  = {P(A)*P(coin_flip):.4f}")
print(f"  P(A∩coin)                     = {P(A & coin_flip):.4f}")
print(f"  Equal?                        = {abs(P(A)*P(coin_flip) - P(A&coin_flip)) < 0.005}  (YES → independent)")

print()
print("Key insight for ML: features that are NOT independent of the target are USEFUL features!")
print("Features that ARE independent of the target are USELESS (they add noise, not signal).")

## Part 3: The Law of Total Probability

### Plain English First

Suppose you want P(house is expensive), but it's easier to compute it **separately for each bedroom count** and then combine.

If you divide all houses into groups (1 bed, 2 beds, 3 beds, ...):  
P(expensive) = P(expensive | 1 bed)×P(1 bed) + P(expensive | 2 beds)×P(2 beds) + ...

**You're taking a weighted average** of the conditional probabilities, weighted by how common each group is.

$$P(A) = \sum_{i} P(A | B_i) \cdot P(B_i)$$

where B₁, B₂, ... are a **partition** of Ω (non-overlapping groups that cover everything).

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Law of Total Probability:
# P(expensive) = Σ P(expensive | bedrooms=k) × P(bedrooms=k)
# ─────────────────────────────────────────────────────────────────

bedroom_counts = [1, 2, 3, 4, 5, 6]

print("LAW OF TOTAL PROBABILITY")
print("P(expensive) = Σ P(expensive | beds=k) × P(beds=k)")
print()
print(f"{'Bedrooms':12} | {'P(beds=k)':12} | {'P(exp|beds=k)':16} | {'Product':12}")
print("-" * 58)

total_probability = 0.0
for k in bedroom_counts:
    group    = (bedrooms == k)                   # houses with exactly k bedrooms
    p_beds_k = P(group)                          # how common is k bedrooms?
    p_exp_given_k = P_given(A, group) if P(group) > 0 else 0.0  # P(expensive | beds=k)
    product  = p_exp_given_k * p_beds_k          # weighted contribution
    total_probability += product

    print(f"  k={k:<9}  | {p_beds_k:12.4f} | {p_exp_given_k:16.4f} | {product:12.4f}")

print("-" * 58)
print(f"  SUM          |              |                  | {total_probability:12.4f}  ← Total Probability")
print()
print(f"Direct P(expensive)   = {P(A):.4f}")
print(f"Via total probability = {total_probability:.4f}")
print(f"Match: {abs(P(A) - total_probability) < 0.001}  ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize the law of total probability as stacked bars
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

probs_beds_k = [P(bedrooms == k) for k in bedroom_counts]
probs_exp_given_k = [P_given(A, bedrooms == k) if P(bedrooms == k) > 0 else 0
                     for k in bedroom_counts]
products = [p * q for p, q in zip(probs_beds_k, probs_exp_given_k)]

x = np.arange(len(bedroom_counts))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(bedroom_counts)))

# Left: P(expensive | beds=k) for each k
bars = axes[0].bar(x, probs_exp_given_k, color=colors, alpha=0.85, edgecolor='black')
axes[0].axhline(P(A), color='red', linestyle='--', linewidth=2,
                label=f'Overall P(expensive) = {P(A):.3f}')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{k} bed' for k in bedroom_counts])
for bar, val in zip(bars, probs_exp_given_k):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', fontsize=9)
axes[0].set_title('P(expensive | bedrooms=k)\nConditional probabilities by bedroom count', fontsize=11)
axes[0].set_ylabel('P(expensive | beds=k)')
axes[0].legend(fontsize=9); axes[0].set_ylim(0, 0.7)

# Right: Contribution of each group to total probability (stacked)
bottom = 0
for k, product, color in zip(bedroom_counts, products, colors):
    bar = axes[1].bar(0, product, bottom=bottom, color=color, alpha=0.85,
                      edgecolor='black', width=0.5)
    axes[1].text(0, bottom + product/2,
                 f'{k} bed\n{product:.4f}', ha='center', va='center', fontsize=9)
    bottom += product

axes[1].axhline(P(A), color='red', linestyle='--', linewidth=2.5,
                label=f'P(expensive)={P(A):.3f}')
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_title(f'Law of Total Probability\nSum of all contributions = {sum(products):.4f}', fontsize=11)
axes[1].set_ylabel('Contribution to P(expensive)')
axes[1].set_xticks([])
axes[1].legend(fontsize=9)

plt.suptitle('Law of Total Probability: P(expensive) = Σ P(expensive|beds) × P(beds)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## ✅ Self-Check Questions

**1.** P(house is expensive) = 0.25. P(house is large) = 0.30. P(expensive AND large) = 0.15.  
   What is P(expensive | large)? Show the calculation.

**2.** In your housing dataset, P(expensive | 5 bedrooms) = 0.65 and P(expensive) = 0.25.  
   Does knowing a house has 5 bedrooms provide useful information for predicting price? Why?

**3.** Two features: `sqft` and `price`. They have high correlation.  
   Are these features likely to be independent? What does this mean for your model?

**4.** You split your housing data into three city types: urban (P=0.5), suburban (P=0.35), rural (P=0.15).  
   P(expensive | urban)=0.40, P(expensive | suburban)=0.20, P(expensive | rural)=0.05.  
   What is P(expensive) overall? Use the Law of Total Probability.

**5.** A Naive Bayes classifier for house price assumes features (sqft, beds, age, location) are  
   **conditionally independent** given the price class. Is this realistic? Does it matter?

<details>
<summary>Click to see answers</summary>

1. P(expensive | large) = P(expensive AND large) / P(large) = **0.15 / 0.30 = 0.50**.

2. **Yes** — P(expensive | 5 beds) = 0.65 vs P(expensive) = 0.25. This is a big difference! Knowing the house has 5 bedrooms makes it much more likely to be expensive. This feature has high **predictive power**.

3. They are very likely **NOT independent** — larger houses cost more. P(expensive | large sqft) >> P(expensive). In models, this means these features are correlated and may cause collinearity issues in linear regression.

4. P(expensive) = 0.40×0.5 + 0.20×0.35 + 0.05×0.15 = **0.20 + 0.07 + 0.0075 = 0.2775**.

5. **Not realistic** — sqft and beds are highly correlated. But it often **doesn't matter much** in practice: Naive Bayes still performs surprisingly well even when independence is violated, because the ranking of probabilities is often preserved.

</details>

---

## 📌 Summary

| Concept | Formula | ML use |
|---------|---------|--------|
| **Conditional prob** | P(A\|B) = P(A∩B)/P(B) | P(class\|features) — every classifier output |
| **Independence** | P(A∩B) = P(A)·P(B) | Naive Bayes, i.i.d. data assumption |
| **Total probability** | P(A) = Σ P(A\|Bᵢ)·P(Bᵢ) | Marginalizing over hidden variables |

**➡️ Next Notebook:** Bayes' Theorem — the single most important formula for ML, showing how to update beliefs with evidence.